# Chinese Product Title Classification with BERT

Fine-tunes `bert-base-chinese` to classify e-commerce product titles into **715 fine-grained categories** (27 top-level × subcategories), then applies the model to an unlabeled product list.

- **Training data:** Shopee product titles with top-level (大類) and subcategory (小類) labels
- **Model:** `BertForSequenceClassification` (bert-base-chinese), 10 epochs, AdamW lr=2e-5
- **Result:** 78.9% test accuracy on the 715-class subcategory task

> Originally a group course project (midterm, Group 8), run on Google Colab with a T4 GPU. Datasets are not included — see `data/README.md`.


## 1. Setup

Install dependencies with `pip install -r requirements.txt`. A GPU is strongly recommended for training.

In [ ]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np
import pandas as pd

## 2. Load Shopee training data

In [ ]:
from pathlib import Path

DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

shopee_train_df = pd.read_csv(DATA_DIR / "shopee_item_category.csv")
shopee_train_df

# 新增一個新的欄位“大小合併”，將大類和小類合併到同一個欄位
shopee_train_df['大小合併'] = ''

for index, row in shopee_train_df.iterrows():
    shopee_train_df.at[index, '大小合併'] = f"{row['大類']}/{row['小類']}"

shopee_train_df

,商品名稱,大類,小類,大小合併
0,PHILIPS 飛利浦 全自動義式咖啡機-EP3246 (金)+湛盧咖啡豆券8張(24包),家用電器,咖啡機與周邊,家用電器/咖啡機與周邊
1,PHILIPS 飛利浦 小旋風電動洗鞋機 (GCA1000),家用電器,其他家電,家用電器/其他家電
2,PHILIPS 飛利浦 2021旗艦款八合一乾濕兩用拔刮美體刀 除毛刀 BRE740,美妝保養,除毛器材,美妝保養/除毛器材
3,PHILIPS 飛利浦 電鬍刀刀頭 RQ11,美妝保養,電動刮鬍刀,美妝保養/電動刮鬍刀
4,PHILIPS 飛利浦 全自動義式咖啡機-EP3246 (金),家用電器,咖啡機與周邊,家用電器/咖啡機與周邊
...,...,...,...,...
253280,白底青平安扣 贈精美中國繩乙入 天然緬甸硬玉A貨【文華珠寶翡翠專賣店】,愛好與收藏品,原石水晶,愛好與收藏品/原石水晶
253281,Q版可愛牛牛玉墜 天然緬甸硬玉A貨 贈中國繩乙入【文華珠寶翡翠專賣店】,愛好與收藏品,原石水晶,愛好與收藏品/原石水晶
253282,冰種平安扣 起螢放光 購買即贈中國繩乙入 天然緬甸硬玉A貨【文華珠寶翡翠專賣店】,愛好與收藏品,原石水晶,愛好與收藏品/原石水晶
253283,冰種觀音玉墜 贈精美中國繩乙條 天然緬甸硬玉A貨【文華珠寶翡翠專賣店】,愛好與收藏品,原石水晶,愛好與收藏品/原石水晶


In [ ]:
big_categories = shopee_train_df["大類"].unique()
print(big_categories)
print(len(big_categories))

['家用電器' '美妝保養' '保健' '時尚配件' '嬰幼兒童裝童鞋' '手機平板與周邊' '電腦與周邊配件' '汽車類' '影音' '居家生活'
 '男生包包' '戶外與運動用品' '美食、伴手禮' '女生包包/精品' '男生衣著' '女生衣著' '愛好與收藏品' '文具、美術用具'
 '母嬰用品' '機車類' '旅行相關用品/行李箱' '相機&空拍機' '女鞋' '男鞋' '寵物' '書籍及雜誌期刊' '手錶']
27


In [ ]:
small_categories = shopee_train_df['大小合併']
small_categories = small_categories.drop_duplicates().reset_index()
num_small_categories = small_categories.shape[0]
small_categories


,index,大小合併
0,0,家用電器/咖啡機與周邊
1,1,家用電器/其他家電
2,2,美妝保養/除毛器材
3,3,美妝保養/電動刮鬍刀
4,6,家用電器/空氣清淨機
...,...,...
710,133375,手機平板與周邊/電池
711,135881,手錶/其他
712,158016,嬰幼兒童裝童鞋/項鍊
713,159373,嬰幼兒童裝童鞋/皮帶


In [ ]:
# 使用 for 迴圈將欄位「大小合併」的內容替換為對應的數字標籤，並填入 label_mapping 中
label_mapping = {}
for index, row in small_categories.iterrows():
    label_mapping[row['大小合併']] = label_mapping.get(row['大小合併'], len(label_mapping))
# print(label_mapping)

# 資料前處理
### 去除商品名稱多餘文字


In [ ]:
# 去除括弧及括弧內文字
commodity_name = shopee_train_df['商品名稱'].str.replace(r'\【.*?\】|\(.*?\)|\（.*?\）', '', regex=True)
shopee_train_df['商品名稱'] = commodity_name
# 去除結尾的型號、尺寸等英數字
# 註：字元集合中的 (cm) 實際上代表字元 c、m、(、)，因此結尾的小寫 c/m 也會被移除；
# 為了保留原始實驗結果，此處維持原本的規則。
commodity_name = shopee_train_df['商品名稱'].str.replace(r'[A-Z0-9(cm)\/\(\)\s\-\.]+$', '', regex=True)
shopee_train_df['商品名稱'] = commodity_name
shopee_train_df

,商品名稱,大類,小類,大小合併
0,PHILIPS 飛利浦 全自動義式咖啡機-EP3246 +湛盧咖啡豆券8張,家用電器,咖啡機與周邊,家用電器/咖啡機與周邊
1,PHILIPS 飛利浦 小旋風電動洗鞋機,家用電器,其他家電,家用電器/其他家電
2,PHILIPS 飛利浦 2021旗艦款八合一乾濕兩用拔刮美體刀 除毛刀,美妝保養,除毛器材,美妝保養/除毛器材
3,PHILIPS 飛利浦 電鬍刀刀頭,美妝保養,電動刮鬍刀,美妝保養/電動刮鬍刀
4,PHILIPS 飛利浦 全自動義式咖啡機,家用電器,咖啡機與周邊,家用電器/咖啡機與周邊
...,...,...,...,...
253280,白底青平安扣 贈精美中國繩乙入 天然緬甸硬玉A貨,愛好與收藏品,原石水晶,愛好與收藏品/原石水晶
253281,Q版可愛牛牛玉墜 天然緬甸硬玉A貨 贈中國繩乙入,愛好與收藏品,原石水晶,愛好與收藏品/原石水晶
253282,冰種平安扣 起螢放光 購買即贈中國繩乙入 天然緬甸硬玉A貨,愛好與收藏品,原石水晶,愛好與收藏品/原石水晶
253283,冰種觀音玉墜 贈精美中國繩乙條 天然緬甸硬玉A貨,愛好與收藏品,原石水晶,愛好與收藏品/原石水晶


# 抓取60％資料做訓練及測試

In [ ]:
np.random.seed(5566)

# 設定要抽取的比例
sample_percentage = 0.6

# 計算要抽取的樣本數
num_samples = int(len(shopee_train_df) * sample_percentage)

# 隨機抽樣
random_samples = shopee_train_df.sample(n=num_samples, replace=False, random_state=42)
random_samples

,商品名稱,大類,小類,大小合併
241377,Q MAN - Pokemon 寶可夢 小火龍的火鍋店積木,愛好與收藏品,組裝模型,愛好與收藏品/組裝模型
14491,essence 艾森絲 持久遮瑕膏 自然膚色／淺膚色,美妝保養,遮瑕,美妝保養/遮瑕
24699,威剛 XPG GAMMIX S50LiteCS 512GB M.2 SSD PCIE Gen...,電腦與周邊配件,SSD固態硬碟,電腦與周邊配件/SSD固態硬碟
51314,圖解國富論{新版}2019[88折],書籍及雜誌期刊,其他,書籍及雜誌期刊/其他
228130,Case Logic JAUNT 15.6吋筆電後背包 石墨黑,女生包包/精品,電腦後背包,女生包包/精品/電腦後背包
...,...,...,...,...
203974,LG 樂金 PuriCare Mini 隨身淨 空氣清淨機 嬰兒車必備清淨機,家用電器,空氣清淨機,家用電器/空氣清淨機
166958,native 大童鞋 JEFFERSON 小奶油頭鞋-珍珠薄荷,嬰幼兒童裝童鞋,平底鞋,嬰幼兒童裝童鞋/平底鞋
68604,PRADA 經典三角logo尼龍鉚釘鑲嵌小牛皮背帶斜背腰包 廠商直送,女生包包/精品,側/肩背包,女生包包/精品/側/肩背包
7172,特力屋 多用途高強度接著劑 30ml,居家生活,油漆、塗料,居家生活/油漆、塗料


# 載入預訓練的BERT模型、Tokenizer
### tokenizer: bert-base-chinese
### model: bert-base-chinese

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-chinese')
model = BertForSequenceClassification.from_pretrained('bert-base-chinese', num_labels=num_small_categories)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-chinese and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


# Tokenization和建立模型輸入

In [ ]:
def tokenize_reviews(reviews, labels):
    input_ids = [] # convert tokens to integers (each id represent a unique token)
    # The purpose of the attention mask is to handle sequences of varying lengths.
    # In BERT, input sequences are padded or truncated to a fixed length,
    # and the attention mask helps the model know which parts are actual data and which are padding.
    # e.g. ["I", "love", "NLP"] -> [1,1,1,0,0] with fixed length 5
    attention_masks = []

    for review in reviews:
        encoded_dict = tokenizer.encode_plus (
            review,
            add_special_tokens=True, # [CLS ][ SEP][PAD][UKN]
            max_length=128,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        input_ids.append(encoded_dict['input_ids'])
        attention_masks.append(encoded_dict['attention_mask'])

    input_ids = torch.cat (input_ids, dim=0)
    attention_masks = torch.cat (attention_masks, dim=0)
    labels = torch. tensor (labels)

    return input_ids, attention_masks, labels

### 隨機切分為訓練集和測試集



In [ ]:
# X是特徵（商品名稱）， y是標籤（類別）
random_samples = random_samples.astype(str)

x = random_samples['商品名稱']
y = random_samples[['大小合併','大類']]

# 隨機切分為訓練集和測試集

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [ ]:
# 使用 tolist() 方法將 Series 轉換為 Python List[str]
x_train_list = [str(item) for item in x_train.astype(str).tolist()]

# 針對 DataFrame 的每一列進行處理
y_train_list = [list(map(str, row)) for row in y_train.values]

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

# 將 y_train_list 中的每個標籤取出並對應到數字
mapped_labels = [label_mapping[label[0]] for label in y_train_list]

# Convert data to PyTorch tensors
input_ids, attention_masks, labels = tokenize_reviews(x_train_list, mapped_labels)

# Create DataLoader for training data
dataset = TensorDataset (input_ids, attention_masks, labels)
train_loader = DataLoader (dataset, batch_size=32, shuffle=True)


# 建立模型架構、訓練模型

In [ ]:
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 將模型移動到 GPU
model.to(device)

# 定義優化器和損失函數
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
loss_fn = torch.nn.CrossEntropyLoss()

# 模型訓練
for epoch in range(10):
    model.train()  # 將模型設置為訓練模式
    total_loss = 0.0  # 初始化 total_loss
    progress_bar = tqdm(train_loader, desc=f'Epoch {epoch + 1}', leave=False)  # 建立 tqdm 進度條
    for batch in progress_bar:
        optimizer.zero_grad()
        input_ids, attention_mask, labels = batch
        input_ids, attention_mask, labels = input_ids.to(device), attention_mask.to(device), labels.to(device)
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
        progress_bar.set_postfix({'loss': loss.item()})

    average_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch + 1}, Average Loss: {average_loss}")

Epoch 1, Average Loss: 3.021280752906674


Epoch 2, Average Loss: 1.4580958384510718


Epoch 3, Average Loss: 1.0412933117700252


Epoch 4, Average Loss: 0.8226972889351217


Epoch 5, Average Loss: 0.6732589423166294


Epoch 6, Average Loss: 0.5597413679290759


Epoch 7, Average Loss: 0.46712988331325744


Epoch 8, Average Loss: 0.39439775126643084


Epoch 9, Average Loss: 0.334956597175057


Epoch 10, Average Loss: 0.2847092579888474


## 評估模型

In [ ]:
# 使用 tolist() 方法將 Series 轉換為 Python List[str]
x_test_list = [str(item) for item in x_test.astype(str).tolist()]

# 針對 DataFrame 的每一列進行處理
y_test_list = [list(map(str, row)) for row in y_test.values]

In [ ]:
# 將 y_test_list 中的每個標籤取出並映射到數字，使用同一個 label_mapping
mapped_test_labels = [label_mapping[label[0]] for label in y_test_list]

# Convert data to PyTorch tensors
test_input_ids, test_attention_masks, test_labels = tokenize_reviews(x_test_list, mapped_test_labels)
# Create DataLoader for test data
test_dataset = TensorDataset (test_input_ids, test_attention_masks, test_labels)
test_loader = DataLoader (test_dataset, batch_size=32, shuffle=True)


In [ ]:
# 將模型設置為評估模式
model.eval()

# 用於存儲預測標籤和實際標籤的列表
all_predictions = []
all_labels = []

with torch.no_grad():
    for test_batch in test_loader:
        test_input_ids, test_attention_mask, test_labels = test_batch
        test_input_ids, test_attention_mask, test_labels = test_input_ids.to(device), test_attention_mask.to(device), test_labels.to(device)

        # 進行模型預測
        outputs = model(test_input_ids, attention_mask=test_attention_mask)

        # 獲取預測標籤
        predictions = torch.argmax(outputs.logits, dim=1).cpu().numpy()

        # 保存預測標籤和實際標籤
        all_predictions.extend(predictions)
        all_labels.extend(test_labels.cpu().numpy())

# 計算評估指標，例如準確度
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(all_labels, all_predictions)

print(f"Test Accuracy: {accuracy}")

# print前五項預測
for i in range(5):
    print(f"Example {i + 1}: Predicted Label: {all_predictions[i]}, Actual Label: {all_labels[i]}")


Test Accuracy: 0.7886165487744695
Example 1: Predicted Label: 113, Actual Label: 113
Example 2: Predicted Label: 462, Actual Label: 462
Example 3: Predicted Label: 348, Actual Label: 348
Example 4: Predicted Label: 256, Actual Label: 256
Example 5: Predicted Label: 156, Actual Label: 156


In [ ]:
# 將模型設置為評估模式
model.eval()

# 用於存儲預測標籤和實際標籤的列表
all_predictions = []
all_labels = []
all_texts = []  # Added list to store input texts

with torch.no_grad():
    for test_batch in test_loader:
        test_input_ids, test_attention_mask, test_labels = test_batch
        test_input_ids, test_attention_mask, test_labels = test_input_ids.to(device), test_attention_mask.to(device), test_labels.to(device)

        # 進行模型預測
        outputs = model(test_input_ids, attention_mask=test_attention_mask)

        # 獲取預測標籤
        predictions = torch.argmax(outputs.logits, dim=1).cpu().numpy()

        # 保存預測標籤和實際標籤
        all_predictions.extend(predictions)
        all_labels.extend(test_labels.cpu().numpy())
        all_texts.extend([tokenizer.decode(ids, skip_special_tokens=True) for ids in test_input_ids.cpu().numpy()])

# 計算評估指標，例如準確度
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(all_labels, all_predictions)

print(f"Test Accuracy: {accuracy}")

# 打印前五項預測
for i in range(5):
    predicted_label_index = all_predictions[i]
    actual_label_index = all_labels[i]
    predicted_text = list(label_mapping.keys())[list(label_mapping.values()).index(predicted_label_index)]
    actual_text = list(label_mapping.keys())[list(label_mapping.values()).index(actual_label_index)]
    print(f"Example {i + 1}: Text{all_texts[i]}: Predicted classification: {predicted_text}, Actual classification: {actual_text}")

Test Accuracy: 0.7886165487744695
Example 1: Text青 青 和 紙 生 活 多 功 能 貼 - 旅 行 時 光: Predicted classification: 書籍及雜誌期刊/其他, Actual classification: 文具、美術用具/標籤、貼紙
Example 2: Text龍 戰: Predicted classification: 書籍及雜誌期刊/漫畫, Actual classification: 母嬰用品/其他
Example 3: Text電 力 十 足 多 功 能 電 子 錶 - 銀 框 正 版 宏 崑 公 司 貨: Predicted classification: 手錶/男錶, Actual classification: 手錶/男錶
Example 4: Text男 款 長 袖 上 衣 長 袖 薄 恤: Predicted classification: 戶外與運動用品/運動上著/戶外機能上著, Actual classification: 戶外與運動用品/運動上著/戶外機能上著
Example 5: Text犀 牛 盾 適 用 邊 框 背 蓋 手 機 殼 / 皮 克 斯 - 怪 獸 大 學 - 怪 獸 大 學 校 徽: Predicted classification: 手機平板與周邊/手機保護周邊, Actual classification: 手機平板與周邊/手機保護周邊


# 預測(分類) jambo_4.csv

將訓練好的模型套用在未標記的商品清單上。此資料集為課程提供，未公開，因此以下儲存格未保留輸出。

In [ ]:
# 去除括弧及括弧內文字
jambo_test_df = pd.read_csv(DATA_DIR / "jambo_4.csv")
commodity_name = jambo_test_df['商品名稱'].str.replace(r'\【.*?\】|\(.*?\)|\（.*?\）', '', regex=True)
jambo_test_df['商品名稱'] = commodity_name
commodity_name = jambo_test_df['商品名稱'].str.replace(r'[A-Z0-9(cm)\/\(\)\s\-\.]+$', '', regex=True)
jambo_test_df['商品名稱'] = commodity_name
jambo_test_df

In [ ]:
# 使用 tolist() 方法將 Series 轉換為 Python List[str]
jambo_test_list = [str(item) for item in jambo_test_df['商品名稱'].astype(str).tolist()]

small_categories_list = small_categories['大小合併'].astype(str).tolist()

In [ ]:
def process_data(input_text):
  token = tokenizer.encode_plus (
      input_text,
      add_special_tokens=True,
      max_length=128,
      padding='max_length',
      truncation=True,
      return_attention_mask=True,
      return_tensors='pt',
  )

  input_ids = token['input_ids']
  attention_mask = token['attention_mask']

  return input_ids, attention_mask

def make_prediction(input_text, threshold=0.25):
    processed_data = process_data(input_text)

    # Move tensors to the same device as the model
    input_ids = processed_data[0].to(device)
    attention_mask = processed_data[1].to(device)

    # Forward pass
    with torch.no_grad():  # 禁用梯度計算
        outputs = model(input_ids, attention_mask=attention_mask)

    # Get prediction probabilities and labels
    probabilities = torch.nn.functional.softmax(outputs.logits, dim=1).cpu().numpy()
    predicted_label = torch.argmax(outputs.logits, dim=1).item()

    # Check if the highest probability is below the threshold
    if probabilities[0, predicted_label] < threshold:
        return "UNK"
    else:
        return small_categories_list[predicted_label]



In [ ]:
pred = []
for i in tqdm(range(len(jambo_test_df))):
    pred.append(make_prediction(jambo_test_df['商品名稱'][i]))
jambo_test_df["大類/小類"] = pred
jambo_test_df

# 分類結果

### threshold 設置
1. 若將threshold設0.9，除非商品名稱與類別名稱有高度重疊，否則會有大量商品找不到合適的分類 --> 類別設為[UNK]
2. 我們發現若將threshold設為0.25，僅商品名稱模糊導致無法辨識其類別之商品，類別才會被設為[UNK]，故取0.25

### 內容分析
1. 若商品名稱與類別名稱（如：茶、收納櫃、口罩等）有重疊的字，幾乎都可以正確被分類
2. 即使商品名稱與類別名稱沒有重疊的字，多數商品依舊能被正確分類（如："888 0424 A02-巴西磁石* 1+台灣一條根 痛快噴 *" 被分類至「保健/舒緩用品」、"888 0524 A15-SODA BEAUTY炭酸抗痘潔顏慕絲 *" 被分類至「美妝保養/洗面乳」），因訓練資料包含此資訊
3. 部分商品如"第 031 標台灣製2.4A限定版史努比快充頭一個169元"、"公主 0924 A08-KING KONG金剛王中王充電線*" 即使明顯看得出其為「手機平板與周邊」類別之商品，卻因含有史努比、金剛王等關鍵字被分到「愛好與收藏品/動漫周邊」「愛好與收藏品/公仔」類別
4. 「保健」大類別下有眾多小類別，單看商品名稱難以知道其功效並將其分到最適當的小類別（如："舊三郎  0412 A41-鴕鳥精膠囊 *" 被分至「保健/順暢保養食品」，但最適當的分類應為「保健/機能性食品」




In [ ]:
# 將「大類/小類」轉回大類、小類欄位
# 部分大類本身含有「/」（如「女生包包/精品」），直接用 split('/') 會切錯，
# 因此改用訓練資料建立的對照表查回原本的大類與小類。
category_pairs = (
    shopee_train_df[['大小合併', '大類', '小類']]
    .drop_duplicates('大小合併')
    .set_index('大小合併')
)
jambo_test_df['大類'] = jambo_test_df['大類/小類'].map(category_pairs['大類']).fillna('UNK')
jambo_test_df['小類'] = jambo_test_df['大類/小類'].map(category_pairs['小類'])
jambo_test_df = jambo_test_df.drop('大類/小類', axis=1)
jambo_test_df.to_csv(OUTPUT_DIR / 'jambo_test_df.csv', index=False)
jambo_test_df['大類'].value_counts()
